In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
from pyhydra.climate.spatial_analysis import HierarchicalGEV

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hierarchical Bayesian GEV Model

`HierarchicalGEV` shares GEV parameters across stations through a population
distribution estimated by MCMC (Stan). Stations with short records borrow
strength from the regional signal.

**Stan model structure:**
```
mu_station[s]    ~ Normal(mu_pop, sigma_pop)
sigma_station[s] ~ LogNormal(log(sigma_pop), 0.5)
xi_station[s]    ~ Normal(xi_pop, 0.3)

y[s, n] ~ GEV(mu_station[s], sigma_station[s], xi_station[s])
```

**When to use over classical RFA:**
- Record lengths vary widely across stations
- You want full posterior uncertainty on return levels
- Some stations have very few years (< 15)

```bash
pip install pystan
```

---
## Synthetic dataset

Six stations with different record lengths — some very short (10 years) to
demonstrate the pooling benefit.

In [ ]:
from scipy.stats import genextreme

rng = np.random.default_rng(7)

# Population parameters (shared region)
mu_pop    = 500.0
sigma_pop = 150.0
xi_pop    = 0.12

stations = {
    "A": {"mu": 480,  "sigma": 140, "xi": 0.10, "n": 40},
    "B": {"mu": 520,  "sigma": 160, "xi": 0.14, "n": 35},
    "C": {"mu": 550,  "sigma": 155, "xi": 0.11, "n": 10},   # short record
    "D": {"mu": 460,  "sigma": 145, "xi": 0.13, "n": 8},    # very short
    "E": {"mu": 510,  "sigma": 165, "xi": 0.15, "n": 50},
    "F": {"mu": 490,  "sigma": 138, "xi": 0.09, "n": 12},   # short record
}

data_dict = {}
for name, p in stations.items():
    data_dict[name] = genextreme.rvs(
        -p["xi"], loc=p["mu"], scale=p["sigma"],
        size=p["n"], random_state=rng
    )

# Summary table
summary = pd.DataFrame({
    n: {"n": len(v), "mean": v.mean(), "std": v.std(), "max": v.max()}
    for n, v in data_dict.items()
}).T.round(0)
print(summary)

In [ ]:
# Box plots of annual maxima by station
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(
    [data_dict[n] for n in stations],
    labels=list(stations.keys()),
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.5),
)
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title("Annual maxima — 6 stations (numbers above show record length)", fontsize=11)
for i, (n, p) in enumerate(stations.items(), start=1):
    ax.text(i, ax.get_ylim()[1] * 0.97, f"n={p['n']}", ha="center", fontsize=8, color="navy")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 1. Fit the hierarchical model

In [ ]:
model = HierarchicalGEV(
    T_values=[2, 5, 10, 25, 50, 100, 200, 500],
    n_chains=4,
    n_samples=1000,
)

# Run MCMC (requires pystan)
# model.fit(data_dict)
print("(MCMC fitting requires pystan — pip install pystan)")
print("  model.fit(data_dict)")

In [ ]:
# After fitting:
# posterior_df = model.posterior_summary()
# print("Population parameters:")
# print(posterior_df.loc[["mu_pop", "sigma_pop", "xi_pop"]].round(2))

print("Example posterior summary output:")
demo = pd.DataFrame({
    "mean":  [501.2, 148.7, 0.121],
    "std":   [ 18.4,  12.3, 0.038],
    "q2.5":  [465.0, 125.1, 0.048],
    "q97.5": [537.8, 173.2, 0.196],
}, index=["mu_pop", "sigma_pop", "xi_pop"])
print(demo.round(2))

---
## 2. Return levels with credible intervals

In [ ]:
# rl_df = model.return_levels(credible=0.90)
# print(rl_df.round(0))

# Simulated output for demonstration
T_vals = [2, 5, 10, 25, 50, 100, 200, 500]
demo_rl = pd.DataFrame(
    np.random.default_rng(1).integers(300, 1800, size=(6, len(T_vals) * 3)),
    index=list(stations.keys()),
    columns=[f"T{T}_{s}" for T in T_vals for s in ["median", "lower", "upper"]],
)
# Sort values to make medians sensible
for T in T_vals:
    base = 300 + T * 8 + np.arange(6) * 20
    demo_rl[f"T{T}_median"] = base
    demo_rl[f"T{T}_lower"]  = base * 0.82
    demo_rl[f"T{T}_upper"]  = base * 1.22

median_cols = [f"T{T}_median" for T in T_vals]
print("Return level medians (m³/s):")
demo_rl[median_cols].round(0)

In [ ]:
# Return level plot for a single station (with credible interval)
station_plot = "C"   # short record — most benefit from pooling
medians = demo_rl.loc[station_plot, [f"T{T}_median" for T in T_vals]].values
lowers  = demo_rl.loc[station_plot, [f"T{T}_lower"  for T in T_vals]].values
uppers  = demo_rl.loc[station_plot, [f"T{T}_upper"  for T in T_vals]].values

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(T_vals, lowers, uppers, alpha=0.25, color="steelblue", label="90% CI")
ax.semilogx(T_vals, medians, "b-o", lw=2, ms=5, label="Posterior median")

# Empirical
data_s = np.sort(data_dict[station_plot])
n_s = len(data_s)
emp_prob = (np.arange(1, n_s + 1) - 0.44) / (n_s + 0.12)
ax.semilogx(1.0 / (1.0 - emp_prob), data_s, "ko", ms=6, label="Empirical", zorder=5)

ax.set_xlabel("Return period (years)")
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title(f"Station {station_plot} (n={stations[station_plot]['n']} years) — hierarchical Bayes",
             fontsize=11)
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare T=100 return level across stations
medians_100 = demo_rl["T100_median"].values
lowers_100  = demo_rl["T100_lower"].values
uppers_100  = demo_rl["T100_upper"].values
names       = list(stations.keys())
n_records   = [p["n"] for p in stations.values()]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))
ax.bar(x, medians_100, color="steelblue", alpha=0.7, label="Median T=100")
ax.errorbar(x, medians_100,
            yerr=[medians_100 - lowers_100, uppers_100 - medians_100],
            fmt="none", color="black", capsize=5, lw=1.5)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n(n={nr})" for n, nr in zip(names, n_records)])
ax.set_ylabel("T=100 year return level (m³/s)")
ax.set_title("Hierarchical GEV — T=100 return levels with 90% CI", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Pooling benefit — short vs long records

The key advantage of the hierarchical model: stations with very few observations
(C: 10 years, D: 8 years, F: 12 years) get narrower credible intervals than
they would from an independent at-site fit.

In [ ]:
from pyhydra.climate.spatial_analysis import fit_gev_mle, return_level

# At-site MLE for each station
T_comp = 100
atsite_rl = {}
for name, vals in data_dict.items():
    params = fit_gev_mle(vals)
    atsite_rl[name] = return_level(params, T_comp)

hier_rl  = demo_rl["T100_median"].values
hier_ci  = uppers_100 - lowers_100

print(f"{'Station':>8}  {'n':>4}  {'At-site MLE':>12}  {'Hier. median':>12}  {'Hier. CI width':>14}")
for i, name in enumerate(names):
    print(f"{name:>8}  {stations[name]['n']:>4}  "
          f"{atsite_rl[name]:>12.0f}  {hier_rl[i]:>12.0f}  {hier_ci[i]:>14.0f}")

In [ ]:
# Visualise at-site vs hierarchical
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))
ax.scatter(x - 0.15, [atsite_rl[n] for n in names], s=80, marker="D",
           color="tomato", zorder=4, label="At-site MLE")
ax.scatter(x + 0.15, hier_rl, s=80, marker="o",
           color="steelblue", zorder=4, label="Hierarchical median")
ax.errorbar(x + 0.15, hier_rl,
            yerr=[hier_rl - lowers_100, uppers_100 - hier_rl],
            fmt="none", color="steelblue", capsize=5, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{n} (n={stations[n]['n']})" for n in names], rotation=20)
ax.set_ylabel("T=100 return level (m³/s)")
ax.set_title("At-site MLE vs hierarchical Bayes — T=100 year", fontsize=11)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Trace plot diagnostics (requires pystan)

After fitting, check MCMC convergence by inspecting the trace of population
parameters and R̂ (Gelman-Rubin statistic). All R̂ should be close to 1.0.

In [ ]:
# model.fit(data_dict)
# fit = model._fit

# fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
# for ax, param in zip(axes, ["mu_pop", "sigma_pop", "xi_pop"]):
#     samples = fit[param]   # shape: (n_chains, n_samples)
#     for chain in range(samples.shape[0]):
#         ax.plot(samples[chain], lw=0.5, alpha=0.7)
#     ax.set_ylabel(param)
#     ax.grid(alpha=0.3)
# axes[0].set_title("MCMC trace — population parameters")
# axes[-1].set_xlabel("Iteration")
# plt.tight_layout()
# plt.show()

print("(Trace plot — uncomment after fitting with pystan)")